# Merge & Report (Notebook 4 / 4)

Run this notebook **after** notebooks 01, 02, and 03 have all finished.

It:
1. Loads the three partial results files from Google Drive
2. Merges their `approach_results` into a single combined JSON
3. Generates the interactive HTML report via `HtmlGenerator`
4. Saves the HTML to Drive and offers a download link

### Expected Drive folder contents before running
```
MyDrive/treebranchmarks/woodelfhd_sweep/
  partial_woodelf_hd.json         ← written by notebook 01
  partial_original_woodelf.json   ← written by notebook 02
  partial_shap.json               ← written by notebook 03
```

### Merge strategy
All three partial JSONs share the same mission/task structure — they differ only in
which keys appear in each task's `approach_results` dict.  
The merge is a simple dict union: for every `(mission[i], task_result[j])` triple,
the approach_results from all three files are combined.

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Must match the DRIVE_FOLDER used in notebooks 01–03.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')

PATH_HD       = DRIVE_FOLDER / 'partial_woodelf_hd.json'
PATH_ORIGINAL = DRIVE_FOLDER / 'partial_original_woodelf.json'
PATH_SHAP     = DRIVE_FOLDER / 'partial_shap.json'

MERGED_JSON = DRIVE_FOLDER / 'woodelfhd_depth_sweep_experiment.json'
REPORT_HTML = DRIVE_FOLDER / 'woodelfhd_depth_sweep_experiment.html'

# Sanity check — all three partial files must exist before merging
missing = [p for p in [PATH_HD, PATH_ORIGINAL, PATH_SHAP] if not p.exists()]
if missing:
    raise FileNotFoundError(
        f'The following partial result files are missing — run the corresponding '
        f'notebooks first:\n' + '\n'.join(str(p) for p in missing)
    )
print('All three partial result files found.')

In [ ]:
# ── Step 3: Clone treebranchmarks and install ────────────────────────────────
# woodelf_explainer is needed by treebranchmarks even though this notebook
# only calls HtmlGenerator (which does not import woodelf directly).

TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

In [ ]:
# ── Step 4: Merge the three partial result files ─────────────────────────────
import copy, json

def load_json(path):
    with open(path) as f:
        return json.load(f)

j_hd       = load_json(PATH_HD)
j_original = load_json(PATH_ORIGINAL)
j_shap     = load_json(PATH_SHAP)

# Verify all three have the same number of missions and task_results
for label, j in [('original_woodelf', j_original), ('shap', j_shap)]:
    assert len(j['missions']) == len(j_hd['missions']), \
        f'{label}: mission count mismatch ({len(j["missions"])} vs {len(j_hd["missions"])})'
    for i, (m_ref, m_other) in enumerate(zip(j_hd['missions'], j['missions'])):
        assert len(m_ref['task_results']) == len(m_other['task_results']), \
            f'{label} mission[{i}]: task_result count mismatch'

# Use j_hd as the base; merge approach_results from j_original and j_shap
merged = copy.deepcopy(j_hd)
for i, mission in enumerate(merged['missions']):
    for j, task_result in enumerate(mission['task_results']):
        task_result['approach_results'].update(
            j_original['missions'][i]['task_results'][j]['approach_results']
        )
        task_result['approach_results'].update(
            j_shap['missions'][i]['task_results'][j]['approach_results']
        )

# Persist merged JSON to Drive
with open(MERGED_JSON, 'w') as f:
    json.dump(merged, f, indent=2)

# Summary
all_approaches = set()
for m in merged['missions']:
    for tr in m['task_results']:
        all_approaches.update(tr['approach_results'].keys())

print(f'Merged {len(merged["missions"])} missions')
print(f'Approaches in merged file: {sorted(all_approaches)}')
print(f'Merged JSON saved to: {MERGED_JSON}')

In [ ]:
# ── Step 5: Generate HTML report ─────────────────────────────────────────────
# Copy the merged JSON into the local results/ directory that the Experiment
# object expects, then call generate_html().

import os, shutil, pathlib

os.chdir('/content/treebranchmarks')

local_results_dir = pathlib.Path('results')
local_results_dir.mkdir(exist_ok=True)
local_json = local_results_dir / 'woodelfhd_depth_sweep_experiment.json'

shutil.copy(MERGED_JSON, local_json)

from benchmarks.woodelfhd_depth_sweep_experiment import build_experiment

exp = build_experiment()
html_path = exp.generate_html()

# Copy generated HTML to Drive
shutil.copy(html_path, REPORT_HTML)
print(f'HTML report saved to Drive: {REPORT_HTML}')

In [ ]:
# ── Step 6: Download the HTML report ─────────────────────────────────────────
from google.colab import files
files.download(str(REPORT_HTML))